# 96 — Autopick and export stacked nodal gathers

Reads stacked nodal products from notebook 95, autopicks first arrivals on stacked gathers, and exports RefraPy-ready pick tables.

Outputs/replaces only:
- `nodal_stack_picks`
- `nodal_stack_refapy_exports`

In [1]:
from pathlib import Path
import sqlite3
import json
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read, Stream, Trace, UTCDateTime

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CATALOG_DB.parent.mkdir(parents=True, exist_ok=True)
print("CATALOG_DB:", CATALOG_DB)

OUT_ROOT = PROJECT_ROOT / "nodal_stacked_by_geode_v1" / "autopick_exports"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

COMPONENT = "Z"
PICK_TMIN_S = 0.0
PICK_TMAX_S = 0.8
NOISE_TMIN_S = -0.04
NOISE_TMAX_S = 0.0
STA_S = 0.006
LTA_S = 0.050
THRESHOLD = 4.0
WRITE_TABLES = True
print("OUT_ROOT:", OUT_ROOT)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_geode_v1/autopick_exports


## 1. Load stack tables safely

In [2]:
REQUIRED = ["nodal_stacks", "nodal_stack_files", "nodal_stack_members"]
OWNED = ["nodal_stack_picks", "nodal_stack_refapy_exports"]

if not CATALOG_DB.exists():
    raise FileNotFoundError(CATALOG_DB)

with sqlite3.connect(CATALOG_DB) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)["name"].tolist()
missing = [t for t in REQUIRED if t not in tables]
if missing:
    raise RuntimeError(f"Run 95 first. Missing: {missing}")

conn = sqlite3.connect(CATALOG_DB)
nodal_stacks = pd.read_sql("SELECT * FROM nodal_stacks", conn)
nodal_stack_files = pd.read_sql("SELECT * FROM nodal_stack_files", conn)
nodal_stack_members = pd.read_sql("SELECT * FROM nodal_stack_members", conn)

display(nodal_stacks.head())
display(nodal_stack_files.groupby(["component", "file_type"]).size().reset_index(name="n"))

,stack_id,geode_event_id,geode_survey,line,file_no,source_x_truth_m,source_type,n_candidate_members,n_accepted_members,reference_nodal_event_id,median_xcorr_shift_s,median_xcorr_corrcoef,min_member_time_from_final_s,max_member_time_from_final_s,output_dir,status
0,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,hammer,6,4,T1_N2_Refraction1m_T1_N2_E00008,-0.008,0.928603,-46.990,-0.286,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
1,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,hammer,3,2,T1_N2_Refraction1m_T1_N2_E00011,0.003,0.976702,-21.410,-7.324,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
2,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,hammer,7,7,T1_N2_Refraction1m_T1_N2_E00020,0.026,0.929131,-42.758,0.060,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
3,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,hammer,6,3,T1_N2_Refraction1m_T1_N2_E00046,0.056,0.984687,-46.552,-0.332,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
4,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,T1,3012,96.5,hammer,5,3,T1_N2_Refraction1m_T1_N2_E00054,0.013,0.942910,-29.534,-0.174,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok


,component,file_type,n
0,Z,mseed,150
1,Z,png_wiggle,150
2,Z,segy,150


## 2. Autopick helpers

In [3]:
def moving_average(x, n):
    n = max(1, int(n))
    if len(x) < n:
        return np.full_like(x, np.nan, dtype=float)
    return np.convolve(x, np.ones(n) / n, mode="same")

def pick_trace_first_arrival(tr):
    dt = float(tr.stats.delta)
    data = tr.data.astype(float)
    data = data - np.nanmedian(data)
    env = np.abs(data)
    t0 = tr.stats.starttime - UTCDateTime(0)
    t = t0 + np.arange(tr.stats.npts) * dt

    win = (t >= PICK_TMIN_S) & (t <= PICK_TMAX_S)
    if win.sum() < 10:
        return np.nan, "failed_short"

    nsta = max(1, int(round(STA_S / dt)))
    nlta = max(nsta + 1, int(round(LTA_S / dt)))
    sta = moving_average(env**2, nsta)
    lta = moving_average(env**2, nlta)
    ratio = sta / (lta + 1e-30)

    nwin = (t >= NOISE_TMIN_S) & (t < NOISE_TMAX_S)
    if nwin.sum() > 5:
        base = np.nanmedian(ratio[nwin])
        scale = 1.4826 * np.nanmedian(np.abs(ratio[nwin] - base))
        thr = max(THRESHOLD, base + 6 * scale)
    else:
        thr = THRESHOLD

    idxs = np.where(win & np.isfinite(ratio) & (ratio >= thr))[0]
    if len(idxs) == 0:
        return np.nan, "failed_threshold"

    return float(t[idxs[0]]), "autopick_sta_lta"

def get_trace_x(tr):
    return float(getattr(tr.stats, "receiver_x_m", np.nan))

## 3. Pick stacked gathers

In [4]:
pick_rows = []
error_rows = []

stack_files = nodal_stack_files[
    (nodal_stack_files["component"].astype(str).eq(COMPONENT)) &
    (nodal_stack_files["file_type"].astype(str).eq("mseed"))
].copy()

print("Stack MiniSEED files to pick:", len(stack_files))

for _, row in stack_files.iterrows():
    stack_id = row["stack_id"]
    p = Path(row["file_path"])
    meta = nodal_stacks[nodal_stacks["stack_id"].astype(str).eq(str(stack_id))]
    if meta.empty:
        continue
    meta = meta.iloc[0]
    source_x = pd.to_numeric(meta["source_x_truth_m"], errors="coerce")

    try:
        st = read(str(p))
        for tr in st:
            rx = get_trace_x(tr)
            off = rx - source_x if np.isfinite(source_x) and np.isfinite(rx) else np.nan
            pick_s, status = pick_trace_first_arrival(tr)

            pick_rows.append({
                "stack_id": stack_id,
                "geode_event_id": meta.get("geode_event_id"),
                "geode_survey": meta.get("geode_survey"),
                "line": meta.get("line"),
                "file_no": meta.get("file_no"),
                "source_x_m": source_x,
                "station": tr.stats.station,
                "channel": tr.stats.channel,
                "component": COMPONENT,
                "receiver_x_m": rx if np.isfinite(rx) else None,
                "offset_m": off if np.isfinite(off) else None,
                "pick_time_s": pick_s if np.isfinite(pick_s) else None,
                "phase": "P",
                "picker": "sta_lta_stack",
                "pick_status": status,
                "stack_mseed_path": str(p),
            })
    except Exception as e:
        error_rows.append({"stack_id": stack_id, "file_path": str(p), "error": repr(e), "traceback": traceback.format_exc()})

nodal_stack_picks = pd.DataFrame(pick_rows)
pick_errors = pd.DataFrame(error_rows)
print("Pick rows:", len(nodal_stack_picks))
display(nodal_stack_picks["pick_status"].value_counts(dropna=False) if len(nodal_stack_picks) else pd.Series(dtype=int))

Stack MiniSEED files to pick: 150
Pick rows: 5193


pick_status
failed_threshold    3155
autopick_sta_lta    2038
Name: count, dtype: int64

## 4. Export RefraPy CSV and write database tables

In [5]:
refapy = nodal_stack_picks[nodal_stack_picks["pick_status"].astype(str).str.startswith("autopick")].copy()
keep = ["stack_id", "geode_event_id", "geode_survey", "line", "file_no", "source_x_m",
        "receiver_x_m", "offset_m", "pick_time_s", "phase", "component", "picker", "stack_mseed_path"]
refapy = refapy[[c for c in keep if c in refapy.columns]].copy()
refapy["shot_id"] = refapy["stack_id"] if len(refapy) else []

refapy_dir = OUT_ROOT / "refapy"
refapy_dir.mkdir(parents=True, exist_ok=True)
refapy_csv = refapy_dir / "nodal_stacked_refapy_picks.csv"
refapy.to_csv(refapy_csv, index=False)
nodal_stack_picks.to_csv(OUT_ROOT / "nodal_stack_picks_all.csv", index=False)
pick_errors.to_csv(OUT_ROOT / "nodal_stack_pick_errors.csv", index=False)

nodal_stack_refapy_exports = pd.DataFrame([{
    "export_name": "nodal_stacked_refapy_picks",
    "file_path": str(refapy_csv),
    "n_pick_rows": len(refapy),
    "component": COMPONENT,
}])

if WRITE_TABLES:
    with sqlite3.connect(CATALOG_DB) as conn:
        for tname in OWNED:
            conn.execute(f'DROP TABLE IF EXISTS "{tname}"')
        if nodal_stack_picks is None or len(nodal_stack_picks.columns) == 0:
            nodal_stack_picks = pd.DataFrame(columns=["stack_id", "pick_status"])
        nodal_stack_picks.to_sql("nodal_stack_picks", conn, if_exists="fail", index=False)
        nodal_stack_refapy_exports.to_sql("nodal_stack_refapy_exports", conn, if_exists="fail", index=False)
        conn.commit()

print("RefraPy CSV:", refapy_csv)
print("Wrote 96-owned tables:", OWNED)

RefraPy CSV: /Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_geode_v1/autopick_exports/refapy/nodal_stacked_refapy_picks.csv
Wrote 96-owned tables: ['nodal_stack_picks', 'nodal_stack_refapy_exports']
